# Offshore hydrodynamics and structural response with Python

**Open engineering workflow:** metocean → JONSWAP waves → Airy wave kinematics → Morison loads → jacket structural response → fatigue.

This notebook is a transparent engineering/teaching workflow for offshore oil & gas facilities and shows where **DNV SESAM / SIMA** normally replace simplified open calculations in an industrial design workflow.

> **Important:** Environmental conditions, geometry, coefficients and material data are illustrative and not site-specific design data. This notebook is not a substitute for project design basis, NORSOK/API/ISO checks, validated hydrodynamic/FE models, class/authority requirements or independent verification.

[Open in Colab](https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/offshore/offshore_hydrodynamics_jacket_design.ipynb)

## 1. Workflow and industrial mapping

```text
Metocean (Hs, Tp, direction, wind, current)
        |
        v
JONSWAP / directional sea state
        |
        v
Wave kinematics u(z,t), a(z,t)
        |
        v
Morison member loads
        |
        v
Global jacket load distribution
        |
        v
Structural response (beam FE)
        |
        v
Stress history → rainflow → S-N fatigue
```

| This notebook | Typical industrial replacement / extension |
|---|---|
| JONSWAP + Airy | Project metocean database, Wajac/HydroD/SIMA environment |
| Morison equation | SESAM **Wajac** jacket hydrodynamic loading |
| Small beam FE | SESAM **GeniE + Sestra** full structural model |
| Simplified fatigue | SESAM fatigue workflow, hot-spot SCFs and code checks |
| Time-domain loops | **SIMA** for coupled dynamics / marine operations |
| Python loops | OneWorkflow / SimaPy / OrcFxAPI / project automation |

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.optimize import brentq

g = 9.81
rho_water = 1025.0
rho_air = 1.225
print('Python offshore design workflow ready')

## 2. Illustrative metocean cases

`xarray` is convenient for multidimensional metocean data (time, frequency, direction, location and depth). Values below are illustrative North-Sea-like cases only.

In [ ]:
cases = xr.Dataset(
    data_vars={
        'Hs': ('case', [4.0, 8.0, 12.0]),
        'Tp': ('case', [8.0, 12.0, 15.0]),
        'wave_direction': ('case', [0.0, 45.0, 90.0]),
        'wind_speed': ('case', [18.0, 28.0, 38.0]),
        'current_speed': ('case', [0.5, 0.8, 1.1]),
    },
    coords={'case': ['operating', 'storm', 'extreme']},
)
display(cases.to_dataframe())

## 3. JONSWAP sea state

The spectrum is scaled so that $H_s=4\sqrt{m_0}$ with $m_0=\int S(f)df$. Packages such as `wavespectra` can replace this explicit implementation for measured/hindcast directional spectra.

In [ ]:
def jonswap_spectrum(f, Hs, Tp, gamma=3.3):
    f = np.asarray(f, dtype=float)
    fp = 1.0/Tp
    sigma = np.where(f <= fp, 0.07, 0.09)
    r = np.exp(-0.5*((f-fp)/(sigma*fp))**2)
    S0 = g**2/(2*np.pi)**4 * f**(-5) * np.exp(-1.25*(fp/f)**4) * gamma**r
    m0 = np.trapezoid(S0, f)
    return S0 * (Hs/(4*np.sqrt(m0)))**2

f = np.linspace(0.025, 0.5, 1000)
plt.figure(figsize=(9,5))
for name in cases.case.values:
    Hs = float(cases.Hs.sel(case=name)); Tp = float(cases.Tp.sel(case=name))
    plt.plot(f, jonswap_spectrum(f,Hs,Tp), label=f'{name}: Hs={Hs:.0f} m, Tp={Tp:.0f} s')
plt.xlabel('Frequency [Hz]'); plt.ylabel('S(f) [m²/Hz]'); plt.title('JONSWAP sea states')
plt.grid(alpha=.3); plt.legend(); plt.show()

## 4. Irregular wave realization and Airy kinematics

For random phases, $\eta(t)=\sum_i\sqrt{2S(f_i)\Delta f}\cos(2\pi f_i t+\phi_i)$. Finite-depth Airy theory uses $\omega^2=gk\tanh(kh)$.

In [ ]:
rng = np.random.default_rng(42)
t = np.arange(0.0, 20*60.0, 0.5)
f_wave = np.linspace(0.03, 0.45, 280); df = f_wave[1]-f_wave[0]
Hs = float(cases.Hs.sel(case='storm')); Tp = float(cases.Tp.sel(case='storm'))
S = jonswap_spectrum(f_wave,Hs,Tp); amp = np.sqrt(2*S*df)
phase = rng.uniform(0,2*np.pi,len(f_wave))
eta = np.sum(amp[:,None]*np.cos(2*np.pi*f_wave[:,None]*t[None,:]+phase[:,None]),axis=0)
print(f'Target Hs={Hs:.2f} m; realization 4*std={4*np.std(eta):.2f} m')
plt.figure(figsize=(11,4)); plt.plot(t[:600],eta[:600]); plt.xlabel('Time [s]'); plt.ylabel('eta [m]'); plt.grid(alpha=.3); plt.show()

In [ ]:
water_depth = 120.0
omega = 2*np.pi*f_wave
def wave_number(w,h):
    return brentq(lambda k: g*k*np.tanh(k*h)-w**2,1e-8,max(10.0,20*w**2/g+1))
k = np.array([wave_number(w,water_depth) for w in omega])
def airy_kinematics(z,tt):
    tr = np.cosh(k*(z+water_depth))/np.sinh(k*water_depth)
    th = omega[:,None]*tt[None,:]+phase[:,None]
    u = np.sum((amp*omega*tr)[:,None]*np.cos(th),axis=0)
    a = np.sum((-amp*omega**2*tr)[:,None]*np.sin(th),axis=0)
    return u,a
plt.figure(figsize=(11,5))
for z in [-5,-30,-60,-100]:
    u,_ = airy_kinematics(z,t); plt.plot(t[:300],u[:300],label=f'z={z} m')
plt.xlabel('Time [s]'); plt.ylabel('Wave velocity [m/s]'); plt.grid(alpha=.3); plt.legend(); plt.show()

## 5. Morison loading

For a slender cylinder, $q=\tfrac12\rho C_DD|U|U+\rho C_M\pi D^2\dot U/4$, where $U=u_{wave}+U_{current}$. In design, $C_D$, $C_M$, marine growth, roughness, shielding and current profiles come from the approved basis/standards.

In [ ]:
D=1.5; Cd=1.0; Cm=2.0
U_current=float(cases.current_speed.sel(case='storm'))
u_wave,a_wave=airy_kinematics(-30,t); U=u_wave+U_current
q_drag=.5*rho_water*Cd*D*np.abs(U)*U
q_inertia=rho_water*Cm*np.pi*D**2/4*a_wave
q_total=q_drag+q_inertia
print(f'Peak drag={np.max(np.abs(q_drag))/1e3:.1f} kN/m')
print(f'Peak inertia={np.max(np.abs(q_inertia))/1e3:.1f} kN/m')
print(f'Peak total={np.max(np.abs(q_total))/1e3:.1f} kN/m')
plt.figure(figsize=(11,4)); plt.plot(t[:400],q_total[:400]/1e3); plt.ylabel('Line load [kN/m]'); plt.xlabel('Time [s]'); plt.grid(alpha=.3); plt.show()

## 6. Integrate loads over jacket elevation + topside wind

The submerged structure is discretized into strips. This is intentionally simpler than Wajac, which applies hydrodynamic loading member-by-member in the structural geometry.

In [ ]:
zgrid=np.linspace(-water_depth+2,-2,32); t_resp=t[:1200]
Q=[]
for z in zgrid:
    uw,aw=airy_kinematics(z,t_resp); Ut=uw+U_current
    Q.append(.5*rho_water*Cd*D*np.abs(Ut)*Ut + rho_water*Cm*np.pi*D**2/4*aw)
Q=np.array(Q)
base_shear=np.trapezoid(Q,zgrid,axis=0)
base_moment=np.trapezoid(Q*(zgrid[:,None]+water_depth),zgrid,axis=0)
wind=float(cases.wind_speed.sel(case='storm')); A_top=2500.; Cd_top=1.2; z_top=25.
Fwind=.5*rho_air*Cd_top*A_top*wind**2; Mwind=Fwind*(water_depth+z_top)
print(f'Peak base shear={np.max(np.abs(base_shear))/1e6:.2f} MN')
print(f'Peak base moment={np.max(np.abs(base_moment))/1e9:.2f} GN m')
print(f'Topside wind force={Fwind/1e6:.2f} MN')

## 7. Small beam-FE structural model

To keep the mechanics visible, the jacket is represented as an equivalent Euler-Bernoulli cantilever. This is **not** a detailed jacket model; a project model would use the full 3D geometry, joints, piles/soil and code checks in tools such as GeniE/Sestra.

In [ ]:
E=210e9; Ieq=120.; H=water_depth+z_top; ne=24
x=np.linspace(0,H,ne+1); L=x[1]-x[0]; ndof=2*(ne+1); K=np.zeros((ndof,ndof))
ke=E*Ieq/L**3*np.array([[12,6*L,-12,6*L],[6*L,4*L**2,-6*L,2*L**2],[-12,-6*L,12,-6*L],[6*L,2*L**2,-6*L,4*L**2]])
for e in range(ne):
    dof=[2*e,2*e+1,2*(e+1),2*(e+1)+1]; K[np.ix_(dof,dof)]+=ke
free=np.arange(2,ndof)
def solve_equivalent(V,M):
    F=np.zeros(ndof)
    if abs(V)>1e-9:
        h=np.clip(M/V,0,water_depth); n=int(np.argmin(abs(x-h))); F[2*n]+=V
    F[-2]+=Fwind; u=np.zeros(ndof); u[free]=np.linalg.solve(K[np.ix_(free,free)],F[free]); return u
i=np.argmax(np.abs(base_shear)); disp=solve_equivalent(base_shear[i],base_moment[i])
print(f'Equivalent topside displacement={disp[-2]*1000:.1f} mm')
plt.figure(figsize=(5,7)); plt.plot(disp[0::2]*1000,x); plt.xlabel('Displacement [mm]'); plt.ylabel('Elevation [m]'); plt.grid(alpha=.3); plt.show()

## 8. Fatigue demonstration

A nominal stress proxy is obtained from $\sigma=M/Z$. A real FLS workflow needs joint hot-spot SCFs, appropriate S-N curves, thickness/environment corrections and long-term sea-state probabilities.

In [ ]:
Zeq=60.; stress=(base_moment+Mwind)/Zeq/1e6
def turning_points(y):
    y=np.asarray(y); d=np.diff(y); nz=np.where(d!=0)[0]
    yy=np.r_[y[0],y[nz+1]] if len(nz) else np.array([y[0]])
    if len(yy)<3: return yy
    dd=np.diff(yy); return yy[np.r_[True,dd[:-1]*dd[1:]<0,True]]
def rainflow_ranges(y):
    s=[]; cyc=[]
    for p in turning_points(y):
        s.append(float(p))
        while len(s)>=3:
            r1=abs(s[-2]-s[-3]); r2=abs(s[-1]-s[-2])
            if r2<r1: break
            if len(s)==3: cyc.append((r1,.5)); s.pop(-3)
            else: cyc.append((r1,1.)); del s[-3:-1]
    cyc += [(abs(s[i+1]-s[i]),.5) for i in range(len(s)-1)]
    return cyc
cycles=rainflow_ranges(stress); m=3.; A=1e12
D10=sum(n/(A/ds**m) for ds,n in cycles if ds>0)
print(f'Stress range={np.ptp(stress):.2f} MPa')
print(f'Rainflow cycles/half-cycles={len(cycles)}')
print(f'Illustrative 10-min fatigue damage={D10:.3e} (not a design result)')
plt.figure(figsize=(11,4)); plt.plot(t_resp,stress); plt.xlabel('Time [s]'); plt.ylabel('Nominal stress [MPa]'); plt.grid(alpha=.3); plt.show()

## 9. SESAM / SIMA mapping

### Fixed jacket
```text
Project metocean → Wajac → GeniE model → Sestra → joint stresses/SCFs → fatigue + ULS/ALS/SLS
                              └────────────→ Splice / pile-soil
```

### Floater / FPSO / semi
```text
Geometry + mass → HydroD/Wadam/Wasim → SIMA (wind+wave+current+mooring) → motions/loads → SESAM response/fatigue
```

Python is particularly valuable for load-case generation, model variants, orchestration, extraction, envelopes, QC, optimization and reporting. Open-source concept studies can extend this notebook with **Capytaine + MoorPy/MoorDyn**.

## 10. Useful Python extensions

- `xarray`, `pandas`, `netCDF4`: metocean data
- `wavespectra`: directional wave spectra
- `Raschii`: Stokes/Fenton regular-wave kinematics
- `Capytaine`: potential-flow hydrodynamics for floaters
- `MoorPy` / `MoorDyn`: mooring equilibrium/dynamics
- `QATS`: time-series and fatigue processing
- `OpenSeesPy`: detailed educational structural models
- `OpenFAST`: coupled offshore wind
- `OrcFxAPI`: OrcaFlex automation

### Relation to NeqSim
NeqSim is primarily a thermodynamics/process/flow-assurance simulator, not a structural hydrodynamics solver. The useful coupling is at the facility-system boundary: reservoir/wells/process operating cases and equipment information from NeqSim can feed mass, operating and piping/load-case assumptions into environmental and structural design workflows.